# 02 — Graph construction

Walks through how staged tables become Neo4j nodes and edges per `schema/graph_schema.md`.

Prereqs:
* `docker compose up` is running.
* `python -m etl.load_mimic`, `python -m etl.load_nhanes`, `python -m etl.load_pathways` have populated Postgres.
* `python -m etl.build_graph` has written into Neo4j.

In [ ]:
from etl._common import neo4j_session

with neo4j_session() as session:
    counts = session.run(
        '''
        MATCH (n)
        RETURN labels(n)[0] AS label, count(*) AS n
        ORDER BY n DESC
        '''
    ).data()
counts

In [ ]:
with neo4j_session() as session:
    rels = session.run(
        '''
        MATCH ()-[r]->()
        RETURN type(r) AS rel, count(*) AS n
        ORDER BY n DESC
        '''
    ).data()
rels

In [ ]:
# Pull one patient's neighborhood and visualize as a NetworkX graph
import networkx as nx
import matplotlib.pyplot as plt

with neo4j_session() as session:
    rows = session.run(
        '''
        MATCH (p:Patient)
        WITH p LIMIT 1
        MATCH path = (p)-[*1..2]-(m)
        RETURN p.patient_id AS pid, [n IN nodes(path) | labels(n)[0] + ':' + coalesce(toString(n.patient_id), n.name, n.lab_id, n.event_id, n.pathway_id, '?')] AS nodes,
               [r IN relationships(path) | type(r)] AS rels
        LIMIT 100
        '''
    ).data()

G = nx.Graph()
for r in rows:
    for i in range(len(r['rels'])):
        G.add_edge(r['nodes'][i], r['nodes'][i+1], label=r['rels'][i])

plt.figure(figsize=(10, 6))
pos = nx.spring_layout(G, seed=42)
nx.draw(G, pos, with_labels=True, node_size=200, font_size=7)
plt.show()